[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/05_%E3%81%84%E3%82%8D%E3%81%84%E3%82%8D%E3%81%AA%E7%A2%BA%E7%8E%87%E5%88%86%E5%B8%83%E3%81%A8%E3%83%99%E3%82%A4%E3%82%BA.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

# 統計2級-05. いろいろな確率分布とベイズの定理

**できるようになること**
- ポアソン・幾何・一様・指数分布の使い所がわかる
- 各分布の確率を計算できる
- ベイズの定理で確率を更新できる

**前提**：統計3級03（確率分布）　/　**所要**：約30分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

In [ ]:
# ここに書いて試してみよう

## 1. ポアソン分布

「めったに起きないことが、一定時間に何回起きるか」の分布。平均 λ で決まる。
例：1時間の問い合わせ件数、ページの誤植数。
$$ P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad 平均=分散=\lambda $$

> **数IIIメモ**：ポアソン分布・指数分布の式には e（自然対数の底）が出てきますが、**形と使いどころ**が分かれば十分です。確率の値は `scipy` が計算します。

In [ ]:
lam = 3   # 1時間に平均3件
k = np.arange(0, 11)                       # 件数の候補（0〜10）
plt.bar(k, stats.poisson.pmf(k, lam))      # ポアソン分布の確率を棒グラフに
plt.title(f'ポアソン分布 λ={lam}'); plt.xlabel('件数'); plt.show()
print('1時間に0件の確率:', round(stats.poisson.pmf(0, lam), 4))      # ちょうど0件
print('5件以上の確率   :', round(1 - stats.poisson.cdf(4, lam), 4))  # 1−(4件以下)

In [ ]:
# ここに書いて試してみよう

### 二項分布のポアソン近似
$n$ が大きく $p$ が小さいとき、$B(n,p) \approx Poisson(np)$。

In [ ]:
print('B(1000, 0.003) で k=2:', round(stats.binom.pmf(2, 1000, 0.003), 5))  # 二項分布
print('Poisson(3)    で k=2:', round(stats.poisson.pmf(2, 3), 5))           # ポアソン近似（λ=np=3）

In [ ]:
# ここに書いて試してみよう

## 2. 幾何分布

「初めて成功するまでの試行回数」の分布。例：当たるまでくじを引く回数。
平均は $1/p$。

In [ ]:
p = 0.2                                    # 1回の成功確率
k = np.arange(1, 21)                       # 初成功までの回数の候補
plt.bar(k, stats.geom.pmf(k, p))           # 幾何分布の確率を棒グラフに
plt.title(f'幾何分布 p={p}（平均 {1/p:.0f}回）'); plt.show()
print('3回目で初成功:', round(stats.geom.pmf(3, p), 4))  # ちょうど3回目で初成功

In [ ]:
# ここに書いて試してみよう

## 3. 一様分布・指数分布（連続）

- **一様分布**：ある区間で同じ確からしさ（例：0〜1の乱数）
- **指数分布**：次の出来事までの待ち時間（ポアソンと対）。平均 $1/\lambda$

In [ ]:
xs = np.linspace(0, 5, 200)                # 横軸の値
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))  # 1行2列のグラフ
ax[0].plot(xs, stats.uniform.pdf(xs, 0, 2)); ax[0].set_title('一様分布 U(0,2)')   # 左：一様分布
ax[1].plot(xs, stats.expon.pdf(xs, scale=1/1.5)); ax[1].set_title('指数分布 λ=1.5')  # 右：指数分布
plt.show()
# 平均2件/時のとき、次の客まで30分以上空く確率
print('指数分布 P(待ち>0.5h):', round(1 - stats.expon.cdf(0.5, scale=1/2), 4))  # 1−(0.5h以下)

In [ ]:
# ここに書いて試してみよう

## 4. ベイズの定理

「結果」から「原因の確率」を更新する式。
$$ P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} $$

有名な例：検査の的中率。ある病気の有病率1%、検査の感度99%・特異度95%。
「陽性だったとき、本当に病気である確率」は？

> **ひとことメモ**：ベイズの定理は『**新しい情報で確率を更新する**』という考え方が要点。数IIIは不要で、下の式は“かけ算と足し算”だけで計算できます。

In [ ]:
prior = 0.01                 # 有病率 P(病気)
sens = 0.99                  # 感度 P(陽性|病気)
spec = 0.95                  # 特異度 P(陰性|健康)
p_pos = sens * prior + (1 - spec) * (1 - prior)   # P(陽性)＝病気で陽性＋健康で誤陽性
posterior = sens * prior / p_pos                  # ベイズの定理 P(病気|陽性)
print(f'P(陽性) = {p_pos:.4f}')
print(f'P(病気 | 陽性) = {posterior:.3f}')
print('→ 陽性でも実際に病気の確率は約', round(posterior*100), '%（直感より低い！）')

In [ ]:
# ここに書いて試してみよう

> 有病率が低いと、感度が高くても「陽性的中率」は低くなる。
これは検診の解釈で実際に問題になる、ベイズの重要な教訓です。

> **よくある間違い**：有病率が低いと、検査の感度が高くても**陽性的中率は低い**（ベイズの教訓）。「検査で陽性＝ほぼ病気」ではありません。

## 5. 幾何分布の「無記憶性」— 過去は未来に影響しない

毎回同じ確率で起こる試行では、「すでに何回外したか」は次にいつ当たるかに影響しません。これを**無記憶性**といいます（ギャンブラーの誤謬の数理的な否定）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
p = 0.2
def trials_to_success():
    n = 1
    while rng.random() >= p: n += 1
    return n
fresh = [trials_to_success() for _ in range(30000)]
cond = [n-5 for n in (trials_to_success() for _ in range(30000)) if n > 5]  # 5回失敗後の追加回数
print(f'最初からの平均回数        : {np.mean(fresh):.2f}  (理論 1/p = {1/p:.1f})')
print(f'5回失敗した後の追加回数平均: {np.mean(cond):.2f}  → ほぼ同じ（無記憶性）')

In [ ]:
# ここに書いて試してみよう

> 「5回連続ハズレたから次は当たりやすい」は誤り（**ギャンブラーの誤謬**）。毎回独立なら、これまでの結果は次の確率を変えません。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
from scipy import stats
# Q1: 平均3件のポアソンで「0件」の確率を ans に
ans = None   # 例: stats.poisson.pmf(0, 3)
_check('Q1 P(X=0)', ans, stats.poisson.pmf(0, 3))

In [ ]:
from scipy import stats
# Q2: 成功率0.2で「3回目に初成功」の確率を ans に
ans = None   # 例: stats.geom.pmf(3, 0.2)
_check('Q2 幾何 P(X=3)', ans, stats.geom.pmf(3, 0.2))

In [ ]:
# Q3: 有病率0.01・感度0.99・特異度0.95。P(病気|陽性) を ans に
ans = None   # .99*.01 / (.99*.01 + .05*.99)
_check('Q3 陽性的中率', ans, .99*.01/(.99*.01 + .05*.99))

In [ ]:
# Q4: 毎回の成功確率0.25の試行で、初めて成功するまでの回数の期待値(=1/p)を ans に
ans = None   # 例: 1/0.25
_check('Q4 初成功までの期待回数', ans, 1/0.25)

---
## 練習問題

**問1.** あるサイトのエラーは平均して1日2件（ポアソン）。
1日に0件の確率と、4件以上の確率を求めよう。

**問2.** 成功率10%のガチャ。初当たりが5回目になる確率（幾何分布）と、平均回数を求めよう。

**問3.** 迷惑メールは全体の40%。『無料』を含むメールのうち迷惑は80%、
通常メールで『無料』を含むのは10%。
『無料』を含むメールが迷惑メールである確率をベイズの定理で求めよう。

**問4.** 成功確率 p=0.1 の試行で「初成功までの回数」をシミュレーションし、平均が 1/p=10 に近づくことを確かめよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[05_いろいろな確率分布とベイズ の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/05_%E3%81%84%E3%82%8D%E3%81%84%E3%82%8D%E3%81%AA%E7%A2%BA%E7%8E%87%E5%88%86%E5%B8%83%E3%81%A8%E3%83%99%E3%82%A4%E3%82%BA.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| ポアソン分布 | まれな事象の件数 |
| 幾何分布 | 初成功までの回数 |
| 一様分布 | 区間で等確率 |
| 指数分布 | 次の出来事までの待ち時間 |
| ベイズの定理 | 情報で確率を更新 |

In [ ]:
# チートシート（分布・ベイズ）
from scipy import stats
stats.poisson.pmf(2, 3)            # ポアソン
stats.geom.pmf(3, 0.2)             # 幾何
stats.expon.cdf(0.5, scale=1/2)   # 指数
# ベイズ: P(A|B) = P(B|A)P(A) / P(B)

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "04_統計_2級/05_いろいろな確率分布とベイズ"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")